# Module 05 — Notebook 3 Solutions: Heatmaps and Subplots

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_length
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
%matplotlib inline
df = pd.read_csv(Path("../../../data/synthetic/evaluation_results.csv"))

## Exercise 1 Solution

In [ ]:
# Build per-family pivots (task as index)
piv_a_v1 = df[df["model"]=="model-a-v1"].set_index("task")["score"]
piv_a_v2 = df[df["model"]=="model-a-v2"].set_index("task")["score"]
piv_b_v1 = df[df["model"]=="model-b-v1"].set_index("task")["score"]
piv_b_v2 = df[df["model"]=="model-b-v2"].set_index("task")["score"]

diff_pivot = pd.DataFrame({
    "model-a": piv_a_v2 - piv_a_v1,
    "model-b": piv_b_v2 - piv_b_v1,
}).T   # transpose so rows = families, cols = tasks

# model-a honesty: v2=0.91 minus v1=0.88 = 0.03
a_honesty_gain = round(float(diff_pivot.loc["model-a", "honesty_calibration"]), 2)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(
    diff_pivot, annot=True, fmt=".2f",
    cmap="RdBu", center=0,
    linewidths=0.5, ax=ax
)
ax.set_title("v2 − v1 Score Difference (blue = improved, red = regressed)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
check_type(diff_pivot, pd.DataFrame, "diff_pivot is a DataFrame")
check_equal(diff_pivot.shape, (2, 5), "diff_pivot is 2×5")
check_approx(a_honesty_gain, 0.03, 1e-2, "a_honesty_gain")

## Exercise 2 Solution

In [ ]:
pivot    = df.pivot_table(index="model", columns="task", values="score")
per_model = df.groupby("model")["score"].mean().sort_values(ascending=False)
per_task  = df.groupby("task")["score"].mean().sort_values()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
n_axes = axes.size   # 4

# Top-left: per-model bar
axes[0, 0].bar(per_model.index, per_model.values, color="steelblue", alpha=0.85)
axes[0, 0].set_title("Mean Score by Model")
axes[0, 0].set_ylabel("Mean Score")
axes[0, 0].tick_params(axis="x", rotation=15)

# Top-right: per-task barh
axes[0, 1].barh(per_task.index, per_task.values, color="steelblue", alpha=0.85)
axes[0, 1].set_title("Mean Score by Task")
axes[0, 1].set_xlabel("Mean Score")

# Bottom-left: boxplot by model
sns.boxplot(data=df, x="model", y="score", palette="muted", ax=axes[1, 0])
axes[1, 0].set_title("Score Distribution by Model")
axes[1, 0].tick_params(axis="x", rotation=15)

# Bottom-right: heatmap
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=0.5, ax=axes[1, 1])
axes[1, 1].set_title("Scorecard")
axes[1, 1].tick_params(axis="x", rotation=30)

fig.suptitle("Evaluation Summary", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
check_equal(isinstance(fig, plt.Figure), True, "fig is a Figure")
check_equal(int(n_axes), 4, "figure has 4 axes")